# Train selected final models

This notebook trains the actual model set used by the application. The selected configurations come from `Model Stats Overview.xlsx`: for each algorithm, one configuration has the best overall balance of stroke-risk statistics and one prioritizes a very low false-negative count.

All final models use the weighted data-balancing method and are trained for clinical, lifestyle, and combined feature sets with and without uncertain features. Model files are stored in `ai_module/models`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "ai_module":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.notebook_training import (
    FINAL_MODEL_CONFIGS,
    FINAL_MODEL_DATASET_IDS,
    train_final_models,
)


## Spreadsheet-derived selections

In [ ]:
pd.DataFrame([
    {
        "algorithm": config["algorithm"],
        "profile": config["profile"],
        "threshold": config["classificationThreshold"],
        "source_false_negatives": config["sourceMetrics"]["falseNegatives"],
        "source_false_positives": config["sourceMetrics"].get("falsePositives"),
        "hyperparameters": config["hyperparameters"],
    }
    for config in FINAL_MODEL_CONFIGS
])


## Training settings

`FORCE_RETRAIN = False` reuses existing final models in `ai_module/models`. Set it to `True` when replacing saved model files.

In [ ]:
FORCE_RETRAIN = False
USE_GPU = True

DATASET_IDS = FINAL_MODEL_DATASET_IDS
CONFIGS = FINAL_MODEL_CONFIGS

print(f"Datasets: {len(DATASET_IDS)}")
print(f"Selected configs: {len(CONFIGS)}")
print(f"Final models to process: {len(DATASET_IDS) * len(CONFIGS)}")


In [ ]:
result = train_final_models(
    dataset_ids=DATASET_IDS,
    configs=CONFIGS,
    force_retrain=FORCE_RETRAIN,
    use_gpu=USE_GPU,
)

result["total"], result["trained"], result["reused"], result["modelOutputDir"]


In [ ]:
pd.DataFrame([
    {
        "model_id": model["modelId"],
        "algorithm": model["algorithm"],
        "dataset": model["datasetId"],
        "profile": model["modelId"].split("__final_", 1)[-1],
        "threshold": model["classificationThreshold"],
        "auc": model["metrics"].get("auc"),
        "f1": model["metrics"].get("f1"),
        "precision": model["metrics"].get("precision"),
        "recall": model["metrics"].get("recall"),
        "false_negatives": model["confusionMatrix"].get("fn"),
        "false_positives": model["confusionMatrix"].get("fp"),
        "reused": model["reusedExistingModel"],
    }
    for model in result["models"]
])
